In [ ]:
Задание 3 (вроде бы не все из 3-его задания)

In [ ]:
-- SELECT * FROM simulator_20251220.feed_actions WHERE toDate(time) = today() and action = 'view' ORDER BY time DESC LIMIT 10
/* SELECT uniq(user_id), gender
 FROM simulator_20251220.feed_actions
 group by gender
*/
/* SELECT uniq(user_id) as t, city
 FROM simulator_20251220.feed_actions
group by city
order by t DESC
limit 4
*/
/* select first_value(time)
FROM simulator_20251220.feed_actions
*/
/*
select uniq(user_id) as count_users, age
from simulator_20251220.feed_actions
group by age
order by count_users desc
*/
/*
select count(post_id), toDate(time)
from simulator_20251220.feed_actions
group by toDate(time)
*/
/*
select toStartOfHour(time) as t, count(user_id)
from simulator_20251220.feed_actions
where toDate(time) = yesterday() 
group by t
*/
/*
SELECT  count(action), toStartOfWeek(time) as t
FROM simulator_20251220.feed_actions 
WHERE  action = 'view'
group by t
*/
/* 
SELECT  count(action), toStartOfWeek(time) as t
FROM simulator_20251220.feed_actions 
WHERE  action = 'like'
group by t
*/
SELECT
  user_id
from
  (
    SELECT
      user_id,
      time
    from
      simulator_20251220.feed_actions
  ) t1
  join (
    SELECT
      user_id
    from
      simulator_20251220.message_actions
  ) t2 using user_id --WHERE time between '2025-12-13' and '2025-12-14'
  --limit 1000
  -------------------------------------------------------------------------------------
SELECT
  date,
  count(user_id) AS users
FROM
  (
    SELECT
      DISTINCT user_id,
      toDate(time) AS date
    FROM
      simulator_20251220.feed_actions
    WHERE
      user_id in (
        SELECT
          DISTINCT user_id
        FROM
          (
            SELECT
              user_id,
              min(toDate(time)) AS start_day
            FROM
              simulator_20251220.feed_actions
            GROUP BY
              user_id
            HAVING
              start_day = '2025-10-25'
          )
      )
  )
GROUP BY
  date
SELECT
  start_day,
  day,
  count(user_id) AS users
FROM
  (
    SELECT
      *
    FROM
      (
        SELECT
          user_id,
          min(toDate(time)) AS start_day
        FROM
          simulator_20251220.feed_actions
        GROUP BY
          user_id
      ) t1
      JOIN (
        SELECT
          DISTINCT user_id,
          toDate(time) AS day
        FROM
          simulator_20251220.feed_actions
      ) t2 USING user_id
    WHERE
      start_day >= today() - 10
  )
GROUP BY
  start_day,
  day
  ---------------------------------------------------------------
  ---На первый день пользователей с платного трафика обычно больше, чем с органического.
  ---В долгосрочной перспективе доля удержанных платных пользователей меньше, чем органических.
SELECT
  toString(start_day) as start_day,
  toString(day) as day,
  count(user_id) AS users
FROM
  (
    SELECT
      *
    FROM
      (
        SELECT
          user_id,
          min(toDate(time)) AS start_day
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'ads'
        GROUP BY
          user_id
      ) t1
      JOIN (
        SELECT
          DISTINCT user_id,
          toDate(time) AS day
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'ads'
      ) t2 USING user_id
    WHERE
      start_day >= '2025-10-25'
  )
GROUP BY
  start_day,
  day 
  --------------------------------------------------------------
  --
SELECT
  toString(start_day) as start_day,
  toString(day) as day,
  count(user_id) AS users
FROM
  (
    SELECT
      *
    FROM
      (
        SELECT
          user_id,
          min(toDate(time)) AS start_day
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'organic'
        GROUP BY
          user_id
      ) t1
      JOIN (
        SELECT
          DISTINCT user_id,
          toDate(time) AS day
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'organic'
      ) t2 USING user_id
    WHERE
      start_day >= '2025-10-25'
  )
GROUP BY
  start_day,
  day WITH cohort_data AS (
    -- Находим стартовый день для каждого пользователя из платного трафика
    SELECT
      user_id,
      min(toDate(time)) AS start_day
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'ads'
    GROUP BY
      user_id
  ),
  daily_activity AS (
    -- Находим все дни активности пользователей
    SELECT
      DISTINCT user_id,
      toDate(time) AS activity_day
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'ads'
  ),
  retention_calc AS (
    -- Считаем retention по дням
    SELECT
      cd.start_day,
      da.activity_day,
      (da.activity_day - cd.start_day) AS day_number,
      uniqExact(cd.user_id) AS total_users,
      uniqExactIf(cd.user_id, da.activity_day IS NOT NULL) AS retained_users
    FROM
      cohort_data cd
      LEFT JOIN daily_activity da ON cd.user_id = da.user_id
      AND da.activity_day = cd.start_day + 20 -- конкретно 20 день
    WHERE
      cd.start_day = toDate('2025-10-25') -- 30 когорта (примерная дата)
    GROUP BY
      cd.start_day,
      da.activity_day,
      day_number
  )
SELECT
  start_day,
  activity_day,
  day_number,
  total_users,
  retained_users,
  round(retained_users / total_users * 100, 1) AS retention_percent
FROM
  retention_calc;
WITH cohort_day AS (
    -- Находим дату 30-й когорты
    SELECT
      start_day
    FROM
      (
        SELECT
          min(toDate(time)) AS start_day,
          row_number() OVER (
            ORDER BY
              start_day
          ) AS rn
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'ads'
        GROUP BY
          user_id
      )
    WHERE
      rn = 30
    LIMIT
      1
  ), cohort_users AS (
    -- Все пользователи 30-й когорты
    SELECT
      user_id
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'ads'
      AND toDate(time) = (
        SELECT
          start_day
        FROM
          cohort_day
      )
    GROUP BY
      user_id
  ),
  retained_count AS (
    -- Кто был активен на 20 день
    SELECT
      count(DISTINCT user_id) AS cnt
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'ads'
      AND user_id IN (
        SELECT
          user_id
        FROM
          cohort_users
      )
      AND toDate(time) = (
        SELECT
          start_day
        FROM
          cohort_day
      ) + 20
  )
SELECT
  (
    SELECT
      count()
    FROM
      cohort_users
  ) AS total_users,
  (
    SELECT
      cnt
    FROM
      retained_count
  ) AS retained_users,
  round(
    (
      SELECT
        cnt
      FROM
        retained_count
    ) * 100.0 / NULLIF(
      (
        SELECT
          count()
        FROM
          cohort_users
      ),
      0
    ),
    1
  ) AS retention_percent
FROM
  cohort_day;
-----------------------------------------------------------
  WITH cohort_day AS (
    -- Находим дату 30-й когорты
    SELECT
      start_day
    FROM
      (
        SELECT
          min(toDate(time)) AS start_day,
          row_number() OVER (
            ORDER BY
              start_day
          ) AS rn
        FROM
          simulator_20251220.feed_actions
        WHERE
          source = 'organic'
        GROUP BY
          user_id
      )
    WHERE
      rn = 30
    LIMIT
      1
  ), cohort_users AS (
    -- Все пользователи 30-й когорты
    SELECT
      user_id
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'organic'
      AND toDate(time) = (
        SELECT
          start_day
        FROM
          cohort_day
      )
    GROUP BY
      user_id
  ),
  retained_count AS (
    -- Кто был активен на 20 день
    SELECT
      count(DISTINCT user_id) AS cnt
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'organic'
      AND user_id IN (
        SELECT
          user_id
        FROM
          cohort_users
      )
      AND toDate(time) = (
        SELECT
          start_day
        FROM
          cohort_day
      ) + 20
  )
SELECT
  (
    SELECT
      count()
    FROM
      cohort_users
  ) AS total_users,
  (
    SELECT
      cnt
    FROM
      retained_count
  ) AS retained_users,
  round(
    (
      SELECT
        cnt
      FROM
        retained_count
    ) * 100.0 / NULLIF(
      (
        SELECT
          count()
        FROM
          cohort_users
      ),
      0
    ),
    1
  ) AS retention_percent
FROM
  cohort_day;
------------------------------------
  WITH -- Все когорты платного трафика
  all_cohorts AS (
    SELECT
      user_id,
      min(toDate(time)) AS start_day
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'ads'
      AND toDate(time) >= '2025-10-25' -- запуск сервиса
    GROUP BY
      user_id
  ),
  -- Находим 30-ю когорту (30-й уникальный день после запуска)
  cohorts_ranked AS (
    SELECT
      start_day,
      dense_rank() OVER (
        ORDER BY
          start_day
      ) AS cohort_num
    FROM
      (
        SELECT
          DISTINCT start_day
        FROM
          all_cohorts
      )
    WHERE
      start_day >= '2025-10-25'
  ),
  -- Целевая когорта (30-я)
  target_cohort AS (
    SELECT
      start_day
    FROM
      cohorts_ranked
    WHERE
      cohort_num = 30
    LIMIT
      1
  ), -- Пользователи целевой когорты
  cohort_users AS (
    SELECT
      user_id
    FROM
      all_cohorts
    WHERE
      start_day = (
        SELECT
          start_day
        FROM
          target_cohort
      )
  ),
  -- Retention на 20-й день
  retention_calc AS (
    SELECT
      (
        SELECT
          count()
        FROM
          cohort_users
      ) AS total_users,
      count(DISTINCT fa.user_id) AS retained_users
    FROM
      cohort_users cu
      LEFT JOIN simulator_20251220.feed_actions fa ON cu.user_id = fa.user_id
      AND fa.source = 'ads'
      AND toDate(fa.time) = (
        SELECT
          start_day
        FROM
          target_cohort
      ) + 20
  )
SELECT
  total_users,
  retained_users,
  round(retained_users * 100.0 / total_users, 1) AS retention_percent
FROM
  retention_calc;
-------------------------------------------------
  WITH -- Все когорты органического трафика
  all_cohorts AS (
    SELECT
      user_id,
      min(toDate(time)) AS start_day
    FROM
      simulator_20251220.feed_actions
    WHERE
      source = 'organic'
      AND toDate(time) >= '2025-10-25' -- запуск сервиса
    GROUP BY
      user_id
  ),
  -- Находим 30-ю когорту (30-й уникальный день после запуска)
  cohorts_ranked AS (
    SELECT
      start_day,
      dense_rank() OVER (
        ORDER BY
          start_day
      ) AS cohort_num
    FROM
      (
        SELECT
          DISTINCT start_day
        FROM
          all_cohorts
      )
    WHERE
      start_day >= '2025-10-25'
  ),
  -- Целевая когорта (30-я)
  target_cohort AS (
    SELECT
      start_day
    FROM
      cohorts_ranked
    WHERE
      cohort_num = 30
    LIMIT
      1
  ), -- Пользователи целевой когорты
  cohort_users AS (
    SELECT
      user_id
    FROM
      all_cohorts
    WHERE
      start_day = (
        SELECT
          start_day
        FROM
          target_cohort
      )
  ),
  -- Retention на 20-й день
  retention_calc AS (
    SELECT
      (
        SELECT
          count()
        FROM
          cohort_users
      ) AS total_users,
      count(DISTINCT fa.user_id) AS retained_users
    FROM
      cohort_users cu
      LEFT JOIN simulator_20251220.feed_actions fa ON cu.user_id = fa.user_id
      AND fa.source = 'organic'
      AND toDate(fa.time) = (
        SELECT
          start_day
        FROM
          target_cohort
      ) + 20
  )
SELECT
  total_users,
  retained_users,
  round(retained_users * 100.0 / total_users, 1) AS retention_percent
FROM
  retention_calc;
-------------------------------------------------------
  WITH target_date AS (
    SELECT
      toDate('2025-10-25') + 29 AS cohort_start
  )
SELECT
  round(
    count(
      DISTINCT if(
        toDate(fa2.time) = (
          SELECT
            cohort_start
          FROM
            target_date
        ) + 20,
        fa2.user_id,
        NULL
      )
    ) * 100.0 / count(DISTINCT fa1.user_id),
    1
  ) AS retention_day_20
FROM
  simulator_20251220.feed_actions fa1
  LEFT JOIN simulator_20251220.feed_actions fa2 ON fa1.user_id = fa2.user_id
  AND fa2.source = 'organic'
  AND toDate(fa2.time) = (
    SELECT
      cohort_start
    FROM
      target_date
  ) + 20
WHERE
  fa1.source = 'organic'
  AND toDate(fa1.time) = (
    SELECT
      cohort_start
    FROM
      target_date
  ) -------------------------------------------------------------------------
SELECT
  toString(start_day) AS start_day,
  toString(day) AS day,
  count(user_id) AS users
FROM
  (
    SELECT
      *
    FROM
      (
        SELECT
          user_id,
          min(toDate(time)) AS start_day
        FROM
          simulator_20251220.feed_actions
        GROUP BY
          user_id
      ) t1
      JOIN (
        SELECT
          DISTINCT user_id,
          toDate(time) AS day
        FROM
          simulator_20251220.feed_actions
      ) t2 USING (user_id)
    WHERE
      start_day BETWEEN toDate('2025-12-03') AND toDate('2025-12-10')
  )
GROUP BY
  start_day,
  day
ORDER BY
  start_day,
  day;
  

 ----------------------------------------------------------------------------------
 
SELECT
    toString(start_day) AS start_day,
    toString(day) AS day,
    country,
    os, 
      count(user_id) AS users
FROM
    (
        SELECT
            t1.user_id,
            t1.start_day,
            t1.country,
            t1.os,
            t2.day
        FROM
            (
                SELECT
                    user_id,
                    country,
                    os,
                    min(toDate(time)) AS start_day
                FROM
                    simulator_20251220.feed_actions
                    WHERE source = 'ads'
                GROUP BY
                    user_id, country, os
                    
            ) t1
            JOIN (
                SELECT
                    DISTINCT user_id,
                    toDate(time) AS day
                FROM
                    simulator_20251220.feed_actions
            ) t2 USING (user_id)
        WHERE
            start_day BETWEEN toDate('2025-12-03') AND toDate('2025-12-10')
            
            
    )
GROUP BY
    start_day,
    day,
    country,
    os
ORDER BY
    start_day,
    day,
    country,
    os;
    
    
    --------------------------------------------
    select * from simulator_20251220.feed_actions
    
 ----------------------------------------------------
 SELECT 
    this_week, 
    previous_week, 
    uniq(user_id) as num_users, 
    status 
FROM (
    -- Ушедшие пользователи
    SELECT 
        user_id,
        addWeeks(arrayJoin(weeks_visited), 1) AS this_week,
        arrayJoin(weeks_visited) AS previous_week,
        'gone' AS status
    FROM (
        SELECT 
            user_id,
            groupUniqArray(toMonday(toDate(time))) AS weeks_visited
        FROM simulator_20251220.feed_actions
        GROUP BY user_id
    )
    WHERE NOT has(weeks_visited, addWeeks(arrayJoin(weeks_visited), 1))
    
    UNION ALL
    
    -- Новые и старые пользователи
    SELECT 
        user_id,
        arrayJoin(weeks_visited) AS this_week,
        addWeeks(arrayJoin(weeks_visited), -1) AS previous_week,
        if(
            has(weeks_visited, addWeeks(arrayJoin(weeks_visited), -1)) = 1, 
            'retained', 
            'new'
        ) AS status
    FROM (
        SELECT 
            user_id,
            groupUniqArray(toMonday(toDate(time))) AS weeks_visited
        FROM simulator_20251220.feed_actions
        GROUP BY user_id
    )
)
WHERE this_week != addWeeks(toMonday(today()), 1)
GROUP BY this_week, previous_week, status
ORDER BY this_week, status;
 